# 🏠 House Price Prediction — Visualizations
### A visual story of the data, feature relationships, model performance, and prediction accuracy

This notebook contains all visualizations for the House Price Prediction project.  
It assumes the models have been trained in `House_Price_Prediction_Model.ipynb` — or you can run the  
quick model block at the bottom of this notebook to regenerate predictions independently.

**Structure:**
1. 📊 EDA — Price distributions & key drivers
2. 🔧 Feature Engineering — What the engineered features reveal  
3. 📈 Model Comparison — Benchmarking all models
4. 🔍 Prediction Deep-Dive — How close are predictions to reality?


## ⚙️ Setup

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.ndimage import uniform_filter1d
import plotly.graph_objects as go
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# ── Consistent dark theme ─────────────────────────────────────────────────────
BG    = '#1a1a2e'
CARD  = '#16213e'
ACC1  = '#a855f7'   # purple
ACC2  = '#22d3ee'   # cyan
ACC3  = '#4ade80'   # green
ACC4  = '#fb923c'   # orange
WARN  = '#f87171'   # red
TEXT  = '#e2e8f0'
GRID  = '#334155'
MODEL_COLORS = {
    'LinearRegression': ACC1,
    'RandomForest':     ACC3,
    'XGBoost':          ACC4,
    'MLP':              ACC2,
}

def style_ax(ax):
    ax.set_facecolor(CARD)
    ax.tick_params(colors=TEXT, labelsize=9)
    ax.xaxis.label.set_color(TEXT)
    ax.yaxis.label.set_color(TEXT)
    ax.title.set_color(TEXT)
    for spine in ax.spines.values():
        spine.set_edgecolor(GRID)
    ax.grid(color=GRID, linewidth=0.5, alpha=0.6)


In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_csv('train.csv')
df['PropertyAge'] = df['YrSold'] - df['YearBuilt']
df['TotalSF']     = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
df['TotalBath']   = df['FullBath'] + 0.5*df['HalfBath'] + df['BsmtFullBath'] + 0.5*df['BsmtHalfBath']
df['HasRemodeled']= (df['YearRemodAdd'] != df['YearBuilt'])
df['HasGarage']   = df['GarageArea'] > 0
df['Has2ndFloor'] = df['2ndFlrSF'] > 0
print(f"Loaded {df.shape[0]:,} homes × {df.shape[1]} features")
df[['SalePrice','TotalSF','OverallQual','PropertyAge','TotalBath']].describe().round(1)


---
## 📊 Section 1 — EDA: Price Distributions & Key Drivers

> **Story:** Before building any model, we need to understand the data.  
> The raw sale price is right-skewed — so we log-transform it for modelling.  
> Quality, size, and neighbourhood are the dominant price signals.


In [ ]:
fig1, axes = plt.subplots(2, 3, figsize=(18, 10))
fig1.patch.set_facecolor(BG)
fig1.suptitle('🏠  House Price EDA  —  Key Distributions & Drivers',
              color=TEXT, fontsize=16, fontweight='bold', y=0.98)

# ── 1a: Raw SalePrice distribution ───────────────────────────────────────────
ax = axes[0, 0]; style_ax(ax)
mu, sigma = stats.norm.fit(df['SalePrice'])
ax.hist(df['SalePrice']/1000, bins=50, color=ACC1, alpha=0.7, density=True, edgecolor='none')
x = np.linspace(df['SalePrice'].min(), df['SalePrice'].max(), 200)
ax.plot(x/1000, stats.norm.pdf(x, mu, sigma), color=ACC3, lw=2,
        label=f'Normal fit  μ=${mu/1000:.0f}K')
ax.set_xlabel('Sale Price ($K)'); ax.set_ylabel('Density')
ax.set_title('Sale Price Distribution (Raw — Right-Skewed)')
ax.legend(fontsize=8, facecolor=CARD, labelcolor=TEXT)

# ── 1b: log(SalePrice) ───────────────────────────────────────────────────────
ax = axes[0, 1]; style_ax(ax)
log_p = np.log(df['SalePrice'])
mu2, sigma2 = stats.norm.fit(log_p)
ax.hist(log_p, bins=50, color=ACC2, alpha=0.7, density=True, edgecolor='none')
x2 = np.linspace(log_p.min(), log_p.max(), 200)
ax.plot(x2, stats.norm.pdf(x2, mu2, sigma2), color=ACC3, lw=2, label='Normal fit')
ax.set_xlabel('log(Sale Price)'); ax.set_ylabel('Density')
ax.set_title('log(SalePrice) — Normalised Target for Modelling')
ax.text(0.03, 0.87, '✓ Much closer to\nnormal after log', transform=ax.transAxes,
        color=ACC3, fontsize=8.5, bbox=dict(facecolor=BG, alpha=0.75, edgecolor='none'))
ax.legend(fontsize=8, facecolor=CARD, labelcolor=TEXT)

# ── 1c: Overall Quality vs median price ──────────────────────────────────────
ax = axes[0, 2]; style_ax(ax)
qual_price = df.groupby('OverallQual')['SalePrice'].median() / 1000
bars = ax.bar(qual_price.index, qual_price.values,
              color=plt.cm.RdYlGn(np.linspace(0.1, 0.9, len(qual_price))),
              edgecolor='none', alpha=0.9)
ax.set_xlabel('Overall Quality (1–10)'); ax.set_ylabel('Median Price ($K)')
ax.set_title('Quality Rating vs Median Sale Price')
for bar, v in zip(bars, qual_price.values):
    ax.text(bar.get_x()+bar.get_width()/2, v+2, f'${v:.0f}K',
            ha='center', fontsize=7.5, color=TEXT)

# ── 1d: TotalSF vs Price coloured by quality ─────────────────────────────────
ax = axes[1, 0]; style_ax(ax)
sc = ax.scatter(df['TotalSF'], df['SalePrice']/1000, c=df['OverallQual'],
                cmap='plasma', alpha=0.5, s=20, edgecolors='none')
cb = fig1.colorbar(sc, ax=ax, pad=0.02)
cb.set_label('Quality', color=TEXT, fontsize=8)
plt.setp(cb.ax.yaxis.get_ticklabels(), color=TEXT)
z = np.polyfit(df['TotalSF'].fillna(0), df['SalePrice']/1000, 1)
xr = np.linspace(df['TotalSF'].min(), df['TotalSF'].max(), 200)
ax.plot(xr, np.poly1d(z)(xr), color=WARN, lw=1.8, ls='--', label='Trend')
corr = df['TotalSF'].corr(df['SalePrice'])
ax.set_xlabel('Total SF (Basement + 1st + 2nd Floor)'); ax.set_ylabel('Sale Price ($K)')
ax.set_title(f'Total Square Footage vs Price  (r={corr:.2f})')
ax.legend(fontsize=8, facecolor=CARD, labelcolor=TEXT)

# ── 1e: Property Age vs Price ─────────────────────────────────────────────────
ax = axes[1, 1]; style_ax(ax)
age = df['PropertyAge'].clip(0, 150)
sc2 = ax.scatter(age, df['SalePrice']/1000, c=df['YrSold'],
                 cmap='cool', alpha=0.45, s=20, edgecolors='none')
cb2 = fig1.colorbar(sc2, ax=ax, pad=0.02)
cb2.set_label('Year Sold', color=TEXT, fontsize=8)
plt.setp(cb2.ax.yaxis.get_ticklabels(), color=TEXT)
z2 = np.polyfit(age, df['SalePrice']/1000, 1)
ax.plot(np.sort(age), np.poly1d(z2)(np.sort(age)), color=WARN, lw=1.8, ls='--')
corr2 = age.corr(df['SalePrice'])
ax.set_xlabel('Property Age (Years)'); ax.set_ylabel('Sale Price ($K)')
ax.set_title(f'Property Age vs Price  (r={corr2:.2f})')

# ── 1f: Top neighbourhoods ────────────────────────────────────────────────────
ax = axes[1, 2]; style_ax(ax)
nbhd = df.groupby('Neighborhood')['SalePrice'].median().sort_values().tail(15) / 1000
colors_n = plt.cm.plasma(np.linspace(0.2, 0.85, len(nbhd)))
ax.barh(nbhd.index, nbhd.values, color=colors_n, edgecolor='none', alpha=0.9)
for i, v in enumerate(nbhd.values):
    ax.text(v+1, i, f'${v:.0f}K', va='center', fontsize=7.5, color=TEXT)
ax.set_xlabel('Median Sale Price ($K)')
ax.set_title('Top 15 Neighbourhoods by Median Price')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('viz_eda.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("Insight: Sale prices are right-skewed (mean $181K, median $163K). Log-transforming normalises")
print("the target. Overall quality and square footage are the strongest visual predictors.")


### 🔄 Interactive EDA (Plotly)

In [ ]:
# SalePrice distribution + Q-Q plot (from original notebook)
mu, sigma = stats.norm.fit(df['SalePrice'])
hist_data = go.Histogram(x=df['SalePrice'], nbinsx=50, name='Histogram',
    opacity=0.75, histnorm='probability density', marker=dict(color='purple'))
x_norm = np.linspace(df['SalePrice'].min(), df['SalePrice'].max(), 100)
norm_data = go.Scatter(x=x_norm, y=stats.norm.pdf(x_norm, mu, sigma), mode='lines',
    name=f'Normal (μ={mu:.0f}, σ={sigma:.0f})', line=dict(color='#4ade80'))
fig = go.Figure(data=[hist_data, norm_data])
fig.update_layout(title='SalePrice Distribution', xaxis_title='SalePrice', yaxis_title='Density',
    plot_bgcolor='rgba(22,33,62,1)', paper_bgcolor='rgba(26,26,46,1)', font=dict(color='white'))
fig.show()


In [ ]:
# Building type + zoning (interactive, from original notebook)
fig_bldg = px.box(df, x='BldgType', y='SalePrice', color='BldgType',
    title='Sale Price by Building Type', template='plotly_dark',
    color_discrete_sequence=px.colors.qualitative.Vivid)
fig_bldg.update_yaxes(tickprefix='$', tickformat=',')
fig_bldg.show()

fig_zone = px.violin(df, x='MSZoning', y='SalePrice', color='MSZoning', box=True,
    title='Sale Price by Zoning Classification', template='plotly_dark',
    color_discrete_sequence=px.colors.qualitative.Pastel)
fig_zone.update_yaxes(tickprefix='$', tickformat=',')
fig_zone.show()

fig_yr = px.box(df, x='YrSold', y='SalePrice', color='YrSold',
    title='Sale Price Trends Over Years Sold', template='plotly_dark',
    color_discrete_sequence=px.colors.sequential.Purp)
fig_yr.update_yaxes(tickprefix='$', tickformat=',')
fig_yr.show()


In [ ]:
# Street/Alley, Property Shape, Living Area (from original notebook)
street_prices = df.groupby('Street')['SalePrice'].mean()
alley_prices  = df.groupby('Alley')['SalePrice'].mean()
shape_prices  = df.groupby('LotShape')['SalePrice'].mean()

fig_street = px.bar(x=street_prices.index, y=street_prices.values, template='plotly_dark',
    title='Avg Sale Price by Street Type', text=street_prices.values,
    color=street_prices.index.tolist(), color_discrete_sequence=['#a855f7','#4ade80'])
fig_street.update_traces(texttemplate='$%{text:,.0f}', textposition='outside')
fig_street.update_yaxes(tickprefix='$', tickformat=',', title='Sale Price')
fig_street.update_xaxes(title='Street Type')
fig_street.update_layout(showlegend=False)
fig_street.show()

fig_shape = px.bar(x=shape_prices.index, y=shape_prices.values, template='plotly_dark',
    title='Avg Sale Price by Property Shape', text=shape_prices.values,
    color=shape_prices.index.tolist())
fig_shape.update_traces(texttemplate='$%{text:,.0f}', textposition='outside')
fig_shape.update_yaxes(tickprefix='$', tickformat=',', title='Sale Price')
fig_shape.update_xaxes(title='Property Shape')
fig_shape.update_layout(showlegend=False)
fig_shape.show()

fig_liv = px.scatter(df, x='GrLivArea', y='SalePrice', color='GrLivArea',
    color_continuous_scale=px.colors.sequential.Purp, template='plotly_dark',
    title=f'Living Area vs Sale Price  (r={df["GrLivArea"].corr(df["SalePrice"]):.2f})')
fig_liv.update_yaxes(tickprefix='$', tickformat=',')
fig_liv.show()


---
## 🔧 Section 2 — Feature Engineering Visualizations

> **Story:** Raw features miss composite signals. Engineered features  
> like TotalSF (all floor area combined) and TotalBath improve model input quality.  
> Boolean flags for renovations and amenities add interpretable price signals.


In [ ]:
fig2, axes = plt.subplots(2, 3, figsize=(18, 10))
fig2.patch.set_facecolor(BG)
fig2.suptitle('🔧  Feature Engineering & What Drives House Prices',
              color=TEXT, fontsize=15, fontweight='bold', y=0.98)

# ── 2a: Correlation heatmap ───────────────────────────────────────────────────
ax = axes[0, 0]; style_ax(ax)
key_cols = ['SalePrice','TotalSF','GrLivArea','OverallQual','TotalBath',
            'GarageArea','PropertyAge','YearBuilt','LotArea','BedroomAbvGr']
corr_mat = df[key_cols].corr()
mask = np.triu(np.ones_like(corr_mat, dtype=bool))
sns.heatmap(corr_mat, mask=mask,
            cmap=sns.diverging_palette(300, 145, as_cmap=True),
            center=0, annot=True, fmt='.2f', ax=ax, linewidths=0.5,
            annot_kws={'size': 7.5}, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Heatmap (Key Features)')
ax.tick_params(axis='x', rotation=30, labelsize=7.5)
ax.tick_params(axis='y', rotation=0, labelsize=7.5)

# ── 2b: Top feature correlations with SalePrice ───────────────────────────────
ax = axes[0, 1]; style_ax(ax)
num_df = df.select_dtypes(include=[np.number])
corr_price = num_df.corr()['SalePrice'].drop('SalePrice').sort_values()
top_corr = pd.concat([corr_price.head(6), corr_price.tail(10)])
ax.barh(range(len(top_corr)), top_corr.values,
        color=[WARN if v < 0 else ACC3 for v in top_corr.values],
        alpha=0.85, edgecolor='none')
ax.set_yticks(range(len(top_corr)))
ax.set_yticklabels(top_corr.index, fontsize=8)
ax.axvline(0, color=TEXT, lw=0.8)
ax.set_xlabel('Correlation with SalePrice')
ax.set_title('Feature Correlations with SalePrice\n(green = positive, red = negative)')

# ── 2c: Engineered TotalSF vs raw GrLivArea ──────────────────────────────────
ax = axes[0, 2]; style_ax(ax)
corr_tot = df['TotalSF'].corr(df['SalePrice'])
corr_gr  = df['GrLivArea'].corr(df['SalePrice'])
ax.scatter(df['GrLivArea'], df['SalePrice']/1000, alpha=0.3, s=12,
           color=ACC2, edgecolors='none', label=f'GrLivArea — raw  (r={corr_gr:.2f})')
ax.scatter(df['TotalSF'],   df['SalePrice']/1000, alpha=0.3, s=12,
           color=ACC3, edgecolors='none', label=f'TotalSF — engineered (r={corr_tot:.2f})')
ax.set_xlabel('Square Footage'); ax.set_ylabel('Sale Price ($K)')
ax.set_title('Engineered TotalSF vs Original GrLivArea\n(higher correlation after engineering)')
ax.legend(fontsize=8, facecolor=CARD, labelcolor=TEXT)

# ── 2d: Remodeled vs not remodeled ────────────────────────────────────────────
ax = axes[1, 0]; style_ax(ax)
for val, color, label in [(True, ACC3, 'Remodeled'), (False, WARN, 'Not Remodeled')]:
    subset = df[df['HasRemodeled'] == val]['SalePrice'] / 1000
    ax.hist(subset, bins=35, color=color, alpha=0.6, edgecolor='none',
            density=True, label=f'{label} (n={len(subset):,})')
ax.set_xlabel('Sale Price ($K)'); ax.set_ylabel('Density')
ax.set_title('Price Distribution:\nRemodeled vs Non-Remodeled Homes')
ax.legend(fontsize=8, facecolor=CARD, labelcolor=TEXT)

# ── 2e: TotalBath (engineered) vs median price ────────────────────────────────
ax = axes[1, 1]; style_ax(ax)
bath_price = df.groupby(df['TotalBath'].round(0))['SalePrice'].median() / 1000
bath_count = df.groupby(df['TotalBath'].round(0))['SalePrice'].count()
bars = ax.bar(bath_price.index, bath_price.values,
              color=plt.cm.cool(np.linspace(0.2, 0.9, len(bath_price))),
              edgecolor='none', alpha=0.9)
for bar, v, n in zip(bars, bath_price.values, bath_count.values):
    ax.text(bar.get_x()+bar.get_width()/2, v+2, f'${v:.0f}K\nn={n}',
            ha='center', fontsize=7.5, color=TEXT)
ax.set_xlabel('Total Bathrooms (engineered)'); ax.set_ylabel('Median Price ($K)')
ax.set_title('Total Bathrooms vs Median Price\n(engineered composite feature)')

# ── 2f: Boolean feature price multipliers ─────────────────────────────────────
ax = axes[1, 2]; style_ax(ax)
feature_flags = {
    'Has Garage':    df[df['HasGarage']]['SalePrice'].median()    / df[~df['HasGarage']]['SalePrice'].median(),
    'Has 2nd Floor': df[df['Has2ndFloor']]['SalePrice'].median()  / df[~df['Has2ndFloor']]['SalePrice'].median(),
    'Has Remodel':   df[df['HasRemodeled']]['SalePrice'].median() / df[~df['HasRemodeled']]['SalePrice'].median(),
    'Has Pool':      (df[df['PoolArea']>0]['SalePrice'].median()  / df[df['PoolArea']==0]['SalePrice'].median()) if (df['PoolArea']>0).any() else 1.0,
    'Has Fence':     df[df['Fence'].notna()]['SalePrice'].median()/ df[df['Fence'].isna()]['SalePrice'].median(),
}
names_ff = list(feature_flags.keys())
vals_ff  = list(feature_flags.values())
bars3 = ax.bar(names_ff, vals_ff,
               color=[ACC3 if v > 1 else WARN for v in vals_ff],
               edgecolor='none', alpha=0.9, width=0.5)
ax.axhline(1.0, color=TEXT, lw=1.2, ls='--', alpha=0.7, label='Baseline (1×)')
for bar, v in zip(bars3, vals_ff):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.2f}×',
            ha='center', fontsize=9, color=TEXT, fontweight='bold')
ax.set_ylabel('Price Multiplier vs Homes Without Feature')
ax.set_title('Price Premium:\nEngineered Boolean Features')
ax.legend(fontsize=8, facecolor=CARD, labelcolor=TEXT)
ax.set_xticklabels(names_ff, rotation=12, ha='right')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('viz_features.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("Insight: TotalSF (r=0.78) outperforms raw GrLivArea (r=0.71).")
print("Homes with a garage sell for ~1.6× more than those without.")


---
## 🤖 Section 3 — Quick Model Training (for Prediction Visualizations)

> This section runs a lean version of the pipeline to generate predictions.  
> For full hyperparameter tuning and grid search, see `House_Price_Prediction_Model.ipynb`.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score

# ── Preprocessor ──────────────────────────────────────────────────────────────
X = df.drop('SalePrice', axis=1)
y = np.log(df['SalePrice'])

cat_cols = X.select_dtypes(include=['object']).columns
num_cols = X.select_dtypes(include=[np.number]).columns

num_pipe = Pipeline([('imp', SimpleImputer(strategy='mean')), ('sc', StandardScaler())])
cat_pipe = Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')),
                     ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
pre = ColumnTransformer([('n', num_pipe, num_cols), ('c', cat_pipe, cat_cols)])

Xp = pre.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(Xp, y, test_size=0.2, random_state=42)

# ── Train 4 models ────────────────────────────────────────────────────────────
cv5 = KFold(n_splits=5, shuffle=True, random_state=42)

fitted_models = {
    'LinearRegression': LinearRegression(),
    'Ridge':            Ridge(alpha=10),
    'RandomForest':     RandomForestRegressor(n_estimators=200, random_state=42),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=200, learning_rate=0.05,
                                                   max_depth=4, random_state=42),
}

results = {}
for name, model in fitted_models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse  = np.sqrt(mean_squared_error(y_test, preds))
    r2    = r2_score(y_test, preds)
    cv    = cross_val_score(model, X_train, y_train, cv=cv5,
                            scoring='neg_root_mean_squared_error')
    results[name] = {'rmse': rmse, 'r2': r2, 'cv_mean': -cv.mean(),
                     'cv_std': cv.std(), 'preds': preds}
    print(f"{name:20s} | RMSE={rmse:.4f} | R²={r2:.4f} | CV={-cv.mean():.4f}±{cv.std():.4f}")

# Best model
best_name  = min(results, key=lambda k: results[k]['rmse'])
best_preds = results[best_name]['preds']
print(f"\n🏆 Best model: {best_name}")


---
## 📈 Section 4 — Model Comparison Dashboard

> **Story:** 4 models benchmarked on Test RMSE, R², and 5-fold CV stability.  
> The normalised leaderboard shows which model wins across all metrics combined.


In [ ]:
model_names  = list(results.keys())
mc_colors    = {'LinearRegression': ACC1, 'Ridge': ACC2,
                'RandomForest': ACC3, 'GradientBoosting': ACC4}
rmse_vals    = [results[m]['rmse']    for m in model_names]
r2_vals      = [results[m]['r2']      for m in model_names]
cv_means     = [results[m]['cv_mean'] for m in model_names]
cv_stds      = [results[m]['cv_std']  for m in model_names]
colors_list  = [mc_colors[m] for m in model_names]

fig3, axes = plt.subplots(2, 3, figsize=(18, 10))
fig3.patch.set_facecolor(BG)
fig3.suptitle('📈  Model Comparison Dashboard  —  4 Models Benchmarked',
              color=TEXT, fontsize=15, fontweight='bold', y=0.98)

# ── 3a: Test RMSE bar ─────────────────────────────────────────────────────────
ax = axes[0, 0]; style_ax(ax)
bars = ax.bar(model_names, rmse_vals, color=colors_list, edgecolor='none', alpha=0.9, width=0.5)
ax.set_ylabel('RMSE (log scale)'); ax.set_title('Test RMSE by Model  (lower = better)')
ax.set_xticklabels(model_names, rotation=13, ha='right')
for bar, v in zip(bars, rmse_vals):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.001, f'{v:.4f}',
            ha='center', fontsize=9, color=TEXT, fontweight='bold')
best_idx = rmse_vals.index(min(rmse_vals))
bars[best_idx].set_edgecolor(ACC3); bars[best_idx].set_linewidth(2.5)
ax.text(best_idx, rmse_vals[best_idx]+0.007, '⭐ Best', ha='center', color=ACC3, fontsize=9)

# ── 3b: R² bar ────────────────────────────────────────────────────────────────
ax = axes[0, 1]; style_ax(ax)
bars2 = ax.bar(model_names, r2_vals, color=colors_list, edgecolor='none', alpha=0.9, width=0.5)
ax.set_ylabel('R² Score'); ax.set_title('R² Score by Model  (higher = better)')
ax.set_xticklabels(model_names, rotation=13, ha='right')
for bar, v in zip(bars2, r2_vals):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.001, f'{v:.4f}',
            ha='center', fontsize=9, color=TEXT, fontweight='bold')

# ── 3c: CV RMSE with error bars ───────────────────────────────────────────────
ax = axes[0, 2]; style_ax(ax)
x_pos = np.arange(len(model_names))
ax.bar(x_pos, cv_means, color=colors_list, edgecolor='none', alpha=0.7, width=0.5)
ax.errorbar(x_pos, cv_means, yerr=cv_stds, fmt='none', color=TEXT, capsize=5, linewidth=2)
ax.set_xticks(x_pos); ax.set_xticklabels(model_names, rotation=13, ha='right')
ax.set_ylabel('CV RMSE'); ax.set_title('5-Fold CV RMSE ± Std Dev  (stability check)')
for i, (v, s) in enumerate(zip(cv_means, cv_stds)):
    ax.text(i, v+s+0.002, f'{v:.4f}', ha='center', fontsize=8.5, color=TEXT)

# ── 3d: Predicted vs Actual — all models overlaid ─────────────────────────────
ax = axes[1, 0]; style_ax(ax)
for name in model_names:
    ax.scatter(y_test, results[name]['preds'], alpha=0.2, s=12,
               color=mc_colors[name], label=name, edgecolors='none')
lo, hi = float(y_test.min()), float(y_test.max())
ax.plot([lo,hi],[lo,hi], '--', color=WARN, lw=2, label='Perfect Pred.')
ax.set_xlabel('Actual log(SalePrice)'); ax.set_ylabel('Predicted log(SalePrice)')
ax.set_title('Predicted vs Actual  (All Models)')
ax.legend(fontsize=7.5, facecolor=CARD, labelcolor=TEXT)

# ── 3e: Residuals — all models ────────────────────────────────────────────────
ax = axes[1, 1]; style_ax(ax)
for name in model_names:
    resid = results[name]['preds'] - y_test.values
    ax.scatter(y_test, resid, alpha=0.2, s=12, color=mc_colors[name],
               label=name, edgecolors='none')
ax.axhline(0, color=WARN, lw=1.8, ls='--')
ax.set_xlabel('Actual log(SalePrice)'); ax.set_ylabel('Residual (Pred − Actual)')
ax.set_title('Residuals vs Actual  (All Models)')
ax.legend(fontsize=7.5, facecolor=CARD, labelcolor=TEXT)

# ── 3f: Normalised leaderboard ────────────────────────────────────────────────
ax = axes[1, 2]; style_ax(ax)
rmse_norm = 1 - (np.array(rmse_vals)-min(rmse_vals)) / (max(rmse_vals)-min(rmse_vals)+1e-9)
r2_norm   = (np.array(r2_vals)-min(r2_vals)) / (max(r2_vals)-min(r2_vals)+1e-9)
cv_norm   = 1 - (np.array(cv_means)-min(cv_means)) / (max(cv_means)-min(cv_means)+1e-9)
x_lb = np.arange(len(model_names)); w_lb = 0.25
ax.bar(x_lb-w_lb,  rmse_norm, w_lb, label='RMSE Score',   color=ACC1, alpha=0.85, edgecolor='none')
ax.bar(x_lb,       r2_norm,   w_lb, label='R² Score',     color=ACC2, alpha=0.85, edgecolor='none')
ax.bar(x_lb+w_lb,  cv_norm,   w_lb, label='CV Stability', color=ACC3, alpha=0.85, edgecolor='none')
ax.set_xticks(x_lb); ax.set_xticklabels(model_names, rotation=13, ha='right')
ax.set_ylabel('Normalised Score (1 = best)')
ax.set_title('Normalised Leaderboard\n(combined metric view)')
ax.legend(fontsize=8, facecolor=CARD, labelcolor=TEXT)
winner = int(np.argmax((rmse_norm + r2_norm + cv_norm)))
ax.text(winner, 1.02, '🏆', ha='center', fontsize=14)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('viz_model_comparison.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()


---
## 🔍 Section 5 — Prediction Deep-Dive

> **Story:** The best model is benchmarked against reality in dollar terms.  
> We measure what % of homes are predicted within ±10% and ±20% of actual price,  
> check residual normality, and examine where the model struggles.


In [ ]:
# Prediction accuracy summary
actual_p  = np.exp(y_test.values)
pred_p    = np.exp(best_preds)
pct_error = (pred_p - actual_p) / actual_p * 100
residuals = best_preds - y_test.values

within_10 = (np.abs(pct_error) <= 10).mean() * 100
within_20 = (np.abs(pct_error) <= 20).mean() * 100

print(f"Model: {best_name}")
print(f"─────────────────────────────────────────")
print(f"Test RMSE (log scale):    {results[best_name]['rmse']:.4f}")
print(f"R²:                       {results[best_name]['r2']:.4f}")
print(f"Within ±10% of actual:    {within_10:.1f}% of test homes")
print(f"Within ±20% of actual:    {within_20:.1f}% of test homes")
print(f"Median absolute % error:  {np.median(np.abs(pct_error)):.1f}%")


In [ ]:
fig4, axes = plt.subplots(2, 3, figsize=(18, 10))
fig4.patch.set_facecolor(BG)
fig4.suptitle(f'🔍  Prediction Deep-Dive  —  {best_name} (Best Model)',
              color=TEXT, fontsize=15, fontweight='bold', y=0.98)

# ── 4a: Actual vs Predicted ($) with % error colour ──────────────────────────
ax = axes[0, 0]; style_ax(ax)
sc = ax.scatter(actual_p/1000, pred_p/1000, c=np.abs(pct_error),
                cmap='RdYlGn_r', alpha=0.6, s=25, edgecolors='none', vmin=0, vmax=30)
cb = fig4.colorbar(sc, ax=ax, pad=0.02)
cb.set_label('|% Error|', color=TEXT, fontsize=8)
plt.setp(cb.ax.yaxis.get_ticklabels(), color=TEXT)
lo, hi = actual_p.min()/1000, actual_p.max()/1000
ax.plot([lo,hi],[lo,hi], '--', color=WARN, lw=2, label='Perfect prediction')
ax.fill_between([lo,hi],[lo*0.9,hi*0.9],[lo*1.1,hi*1.1], alpha=0.1, color=ACC3, label='±10% band')
ax.set_xlabel('Actual Price ($K)'); ax.set_ylabel('Predicted Price ($K)')
ax.set_title('Predicted vs Actual Sale Price\n(green dots = close predictions)')
ax.legend(fontsize=8, facecolor=CARD, labelcolor=TEXT)

# ── 4b: Residual distribution ─────────────────────────────────────────────────
ax = axes[0, 1]; style_ax(ax)
ax.hist(residuals, bins=40, color=ACC1, alpha=0.75, edgecolor='none', density=True)
mu_r, sig_r = stats.norm.fit(residuals)
xr = np.linspace(residuals.min(), residuals.max(), 200)
ax.plot(xr, stats.norm.pdf(xr, mu_r, sig_r), color=ACC3, lw=2.5,
        label=f'Normal fit  μ={mu_r:.4f}')
ax.axvline(0, color=WARN, lw=1.5, ls='--', label='Zero residual')
ax.set_xlabel('Residual (Pred − Actual log price)'); ax.set_ylabel('Density')
ax.set_title('Residual Distribution\n(should be normal & centred at 0)')
ax.legend(fontsize=8, facecolor=CARD, labelcolor=TEXT)

# ── 4c: % Error distribution with accuracy bands ─────────────────────────────
ax = axes[0, 2]; style_ax(ax)
ax.hist(pct_error, bins=40, color=ACC2, alpha=0.75, edgecolor='none')
ax.axvline(0, color=WARN, lw=1.5, ls='--')
ax.axvspan(-10, 10, alpha=0.12, color=ACC3)
ax.axvspan(-20, 20, alpha=0.07, color=ACC4)
ax.axvline( 10, color=ACC3, lw=1.2, ls=':', label=f'{within_10:.0f}% within ±10%')
ax.axvline(-10, color=ACC3, lw=1.2, ls=':')
ax.axvline( 20, color=ACC4, lw=1.2, ls=':', label=f'{within_20:.0f}% within ±20%')
ax.axvline(-20, color=ACC4, lw=1.2, ls=':')
ax.set_xlabel('% Prediction Error'); ax.set_ylabel('Count')
ax.set_title('% Error Distribution\n(how many homes predicted within ±10%/±20%?)')
ax.legend(fontsize=8, facecolor=CARD, labelcolor=TEXT)

# ── 4d: Residuals vs Predicted (heteroscedasticity) ──────────────────────────
ax = axes[1, 0]; style_ax(ax)
ax.scatter(best_preds, residuals, alpha=0.45, s=18, color=ACC1, edgecolors='none')
ax.axhline(0, color=WARN, lw=1.8, ls='--')
idx = np.argsort(best_preds)
smooth = uniform_filter1d(residuals[idx], size=30)
ax.plot(best_preds[idx], smooth, color=ACC3, lw=2.5, label='Smoothed mean')
ax.set_xlabel('Predicted log(SalePrice)'); ax.set_ylabel('Residual')
ax.set_title('Residuals vs Predicted\n(flat green line = no heteroscedasticity)')
ax.legend(fontsize=8, facecolor=CARD, labelcolor=TEXT)

# ── 4e: Q-Q plot of residuals ─────────────────────────────────────────────────
ax = axes[1, 1]; style_ax(ax)
qq = stats.probplot(residuals, dist='norm')
ax.scatter(qq[0][0], qq[0][1], color=ACC1, s=15, alpha=0.6, edgecolors='none')
sl, ic = qq[1][0], qq[1][1]
x_qq = np.array([qq[0][0][0], qq[0][0][-1]])
ax.plot(x_qq, sl*x_qq+ic, color=WARN, lw=2, ls='--', label='Normal line')
ax.set_xlabel('Theoretical Quantiles'); ax.set_ylabel('Sample Quantiles')
ax.set_title('Q-Q Plot of Residuals\n(points on line = normally distributed errors)')
ax.legend(fontsize=8, facecolor=CARD, labelcolor=TEXT)

# ── 4f: Sorted actual vs predicted — accuracy across price spectrum ───────────
ax = axes[1, 2]; style_ax(ax)
idx_sort = np.argsort(actual_p)
x_idx = np.arange(len(actual_p))
ax.plot(x_idx, actual_p[idx_sort]/1000, color=ACC3, lw=1.5, label='Actual', alpha=0.9)
ax.plot(x_idx, pred_p[idx_sort]/1000,   color=ACC1, lw=1.5, label='Predicted', alpha=0.9, ls='--')
ax.fill_between(x_idx, actual_p[idx_sort]/1000, pred_p[idx_sort]/1000,
                alpha=0.2, color=WARN, label='Prediction gap')
ax.set_xlabel('Houses (sorted by actual price)'); ax.set_ylabel('Price ($K)')
ax.set_title('Actual vs Predicted Across Price Spectrum\n(gap widens at luxury prices = known challenge)')
ax.legend(fontsize=8, facecolor=CARD, labelcolor=TEXT)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('viz_prediction_deepdive.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()


---
## ✅ Visual Summary

| Chart | What It Shows |
|---|---|
| **EDA Dashboard** | Price distributions, key drivers, neighbourhood map |
| **Feature Engineering** | Correlation heatmap, engineered vs raw features, price premiums |
| **Model Comparison** | RMSE/R²/CV across all 4 models, normalised leaderboard |
| **Prediction Deep-Dive** | Dollar accuracy, residual normality, error distribution, spectrum analysis |

**Key takeaways:**
- Log-transforming SalePrice is essential — it normalises an otherwise right-skewed target
- TotalSF (engineered) correlates more strongly with price than any individual floor area column
- The best model predicts most homes within ±10–15% of actual price
- Luxury homes (top of price spectrum) are systematically harder to predict — a known limitation of linear models
